# 01 — Data Preparation

This notebook prepares the fictitious workforce, training activity, course catalogue, lead, and sales datasets used throughout the hiring-needs analysis.

The preparation workflow:

1. imports the raw Excel files;
2. standardises column names and analytical variables;
3. restructures multi-value and wide-format fields;
4. validates data quality and referential integrity;
5. inspects the resulting datasets;
6. exports the processed data for the subsequent notebooks.

The objective is to create a consistent and validated analytical foundation in which workforce supply, operational activity, and commercial demand can be compared using common course and location identifiers.

In [38]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

## Project configuration

The notebook uses separate directories for raw and processed data. Raw files are preserved in their original format, while the transformed datasets are exported as CSV files for use in the subsequent analytical stages.

In [39]:
RAW_DATA_DIR = Path("../data/raw")

PROCESSED_DATA_DIR = Path("../data/processed")

## Raw data import

Five fictitious Excel datasets are loaded:

- `base_formadores.xlsx`: trainer identifiers and the courses and locations for which each trainer is registered;
- `historico_acoes.xlsx`: historical, ongoing, cancelled, and planned training actions;
- `info_formacoes.xlsx`: course names, areas, reference durations, and catalogue status;
- `leads.xlsx`: weekly commercial interest by course, modality, and training centre;
- `vendas.xlsx`: weekly enrolments by course, modality, and training centre.

Date columns are parsed during import to ensure that subsequent temporal operations use consistent datetime values.

In [40]:
trainers_df = pd.read_excel(RAW_DATA_DIR / "base_formadores.xlsx")

actions_df = pd.read_excel(RAW_DATA_DIR / "historico_acoes.xlsx", parse_dates=["Data inicial", "Data final"])

info_df = pd.read_excel(RAW_DATA_DIR / "info_formacoes.xlsx")

leads_df = pd.read_excel(RAW_DATA_DIR / "leads.xlsx", parse_dates=["Data"])

sales_df = pd.read_excel(RAW_DATA_DIR / "vendas.xlsx", parse_dates=["Data"])

## Reusable cleaning and validation functions

The following functions standardise column names and implement reusable data-quality checks.

Column names are converted to lowercase ASCII `snake_case`, removing accents, spaces, and special characters. The validation functions detect:

- duplicated records;
- missing values;
- negative numeric values;
- empty strings;
- invalid date sequences;
- foreign-key values that do not exist in their corresponding reference datasets.

The functions raise explicit errors when an inconsistency is found, preventing invalid data from silently progressing through the analytical pipeline.

In [41]:
def clean_columns(df):
    df = df.copy()

    def clean(col):
        col = str(col).strip().lower()
        col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
        col = re.sub(r"[^a-z0-9]+", "_", col)
        col = re.sub(r"_+", "_", col).strip("_")

        return col

    df.columns = [clean(col) for col in df.columns]
    
    return df

In [42]:
def validate_duplicated(df, df_name, subset=None):
    duplicated_rows = df[df.duplicated(subset=subset, keep=False)]

    if not duplicated_rows.empty:
        raise ValueError(
            f"Duplicated rows found in {df_name}"
        )

In [43]:
def validate_missing_values(df, df_name="Data Frame"):
    missing_values = df.isna().sum()
    missing_values = missing_values[missing_values > 0]

    if not missing_values.empty:
        raise ValueError(
            f"Missing values found in {df_name}:\n{missing_values}"
        )

In [44]:
def validate_negative_values(df, df_name="Data Frame", cols=None):
    if cols is None:
        cols = df.select_dtypes(include="number").columns.to_list()
    elif isinstance(cols, str):
        cols = [cols]
    
    negative_mask = df[cols].lt(0)

    if negative_mask.any().any():
        negative_rows = df.loc[negative_mask.any(axis=1), cols]

        raise ValueError(
            f"Negative values found in {df_name} "
            f"(columns: {cols}):\n{negative_rows}"
        )

In [45]:
def validate_empty_strings(df, df_name="DataFrame", cols=None):
    if cols is None:
        cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

    elif isinstance(cols, str):
        cols = [cols]

    empty_mask = df[cols].apply(lambda col: col.astype("string").str.strip().eq(""))

    if empty_mask.any().any():
        empty_rows = df.loc[empty_mask.any(axis=1)]

        raise ValueError(
            f"Empty strings found in {df_name} "
            f"(columns: {cols}):\n{empty_rows}"
        )

In [46]:
def validate_date_order(df, start_col, end_col, df_name="DataFrame"):
    invalid_rows = df.loc[df[end_col] < df[start_col]]

    if not invalid_rows.empty:
        raise ValueError(
            f"Invalid date order found in {df_name} "
            f"'{end_col}' cannot be earlier than "
            f"'{start_col}':\n{invalid_rows}"
        )

In [47]:
def validate_foreign_key(df, column, ref_df, ref_column, df_name="DataFrame", ref_name="reference DataFrame"):
    reference_values = ref_df[ref_column]

    valid_mask = df[column].isin(reference_values)

    invalid_mask = ~valid_mask

    invalid_values = df.loc[invalid_mask, column].unique().tolist()

    if invalid_values:
        raise ValueError(
            f"Invalid values found in {df_name}.{column}. "
            f"These values do not exist in "
            f"{ref_name}.{ref_column}: "
            f"{invalid_values}"
        )

## Column standardisation

The column-cleaning function is applied consistently to all imported datasets before individual transformations are performed. This creates a common naming convention and reduces the risk of errors caused by accents, spaces, capitalisation, or inconsistent separators.

In [48]:
trainers_df = clean_columns(trainers_df)
actions_df = clean_columns(actions_df)
info_df = clean_columns(info_df)
leads_df = clean_columns(leads_df)
sales_df = clean_columns(sales_df)

## Dataset transformation

### Trainer eligibility

The original trainer dataset stores multiple courses and training centres in the same row. These fields are separated and exploded into two relational tables:

- `trainers_course_df`: one row for each trainer–course relationship;
- `trainers_local_df`: one row for each trainer–location relationship.

This structure allows the subsequent analyses to determine which trainers are potentially eligible for each course-location combination. Eligibility indicates registration in the trainer pool and does not confirm actual availability or assignment.

In [49]:
trainers_df = trainers_df.rename(columns={
    "id": "trainer_id",
    "cursos": "course_id",
    "centros_de_formacao": "local"
})

trainers_df = trainers_df[["trainer_id", "course_id", "local"]]


trainers_df["course_id"] = (
    trainers_df["course_id"]
    .astype(str)
    .str.split(" - ")
)

trainers_course_df = trainers_df.explode("course_id", ignore_index=True)

trainers_course_df["course_id"] = trainers_course_df["course_id"].str.strip()

trainers_course_df = trainers_course_df[["trainer_id", "course_id"]]

trainers_course_df = trainers_course_df.sort_values(by=["trainer_id", "course_id"], ignore_index=True)

trainers_df["local"] = (
    trainers_df["local"]
    .astype(str)
    .str.split(" - ")
)

trainers_local_df = trainers_df.explode("local", ignore_index=True)

trainers_local_df["local"] = trainers_local_df["local"].str.strip()

trainers_local_df = trainers_local_df[["trainer_id", "local"]]

trainers_local_df = trainers_local_df.sort_values(by=["trainer_id", "local"], ignore_index=True)


### Training actions

The training-action dataset is standardised around the action identifier, course, location, dates, modality, status, and number of learners.

The action identifier follows a structured convention in which:

- `P` represents in-person training;
- `E` represents online training;
- `H` represents hybrid training;
- the second component identifies the course.

These components are extracted to create explicit `modality` and `course_id` variables. The records are then ordered chronologically to support the subsequent operational and simultaneity analyses.

In [50]:
actions_df = actions_df.rename(columns={
    "id": "action_id",
    "local_de_formacao": "local",
    "estado_da_acao": "status",
    "total_de_formandos_alunos": "total_students",
    "data_inicial": "start_date",
    "data_final": "end_date"
})

actions_df = actions_df[
    ["action_id", "local", "start_date", "end_date", "status", "total_students"]
]

parts_actions = actions_df["action_id"].astype(str).str.strip().str.split("_", expand=True)

actions_df["modality"] = parts_actions[0]

modality_map = {"P": "In-person", "E": "Online", "H": "Hybrid"}

actions_df["modality"] = actions_df["modality"].map(modality_map)

actions_df["course_id"] = parts_actions[1]

actions_df = actions_df[
    ["action_id", "course_id", "local", "start_date", "end_date", "modality", "status", "total_students"]
]

actions_df = actions_df.sort_values(by=["start_date", "end_date", "course_id"], ignore_index=True)

### Course catalogue

The course-information dataset provides the reference table for the analytical workflow. It contains one row per course, including its name, professional area, reference duration, and catalogue status.

The original Portuguese catalogue-status values are converted into Boolean values so that active courses can be selected consistently in later notebooks.

In [51]:
info_df = info_df.rename(columns={
    "id_curso": "course_id",
    "nome_curso": "course_name",
    "area_curso": "course_area",
    "carga_horaria_referencia": "total_hours",
    "ativo_catalogo": "active"
})

info_df = info_df[["course_id", "course_name", "course_area", "total_hours", "active"]]

active_map = {"Sim": True, "Não": False}

info_df["active"] = info_df["active"].map(active_map)

info_df = info_df.sort_values(by="course_id", ignore_index=True)

### Commercial leads

The raw lead dataset is stored in wide format, with separate columns representing course and modality combinations. It is reshaped into long format so that each row represents a specific date, location, course, and modality.

Course and modality identifiers are extracted from the original column names. Lead totals are converted to absolute values to correct sign inconsistencies in the fictitious source data, and zero-volume records are removed because they do not represent observed commercial activity.

In [52]:
leads_df = leads_df.rename(columns={
    "data": "date",
    "centro_de_formacao": "local"
})

leads_df = leads_df.drop(columns="semana")

melt_lead_courses = [
    col for col in leads_df.columns
    if col not in ["date", "local"]
]

leads_df = leads_df.melt(
    id_vars=["date", "local"],
    value_vars=melt_lead_courses,
    var_name="course_id",
    value_name="total_leads"
)

leads_df["course_id"] = leads_df["course_id"].str.upper()

parts_leads = leads_df["course_id"].astype(str).str.strip().str.split("_", expand=True)

leads_df["modality"] = parts_leads[0]

leads_df["modality"] = leads_df["modality"].map(modality_map)

leads_df["course_id"] = parts_leads[1]

leads_df = leads_df[["course_id", "local", "modality", "date", "total_leads"]]

leads_df["total_leads"] = leads_df["total_leads"].abs()

leads_df = leads_df.loc[leads_df["total_leads"] > 0]

leads_df = leads_df.sort_values(by=["date", "total_leads", "course_id"], ascending=[True, False, True], ignore_index=True)

### Commercial sales

The sales dataset is standardised using the same course and modality structure applied to the lead data. Course identifiers and delivery modalities are extracted from the original course code.

Sales totals are converted to absolute values to correct sign inconsistencies in the fictitious source data. Records without enrolments are removed, leaving only observed sales activity for the commercial analysis.

In [53]:
sales_df = sales_df.rename(columns={
    "data": "date",
    "centro_de_formacao": "local",
    "codigo_do_curso": "course_id",
    "inscricoes": "total_sales"
})

parts_sales = sales_df["course_id"].astype(str).str.strip().str.split("_", expand=True)

sales_df["course_id"] = parts_sales[1]

sales_df["modality"] = parts_sales[0]

sales_df["modality"] = sales_df["modality"].map(modality_map)

sales_df = sales_df[["course_id", "local", "modality", "date", "total_sales"]]

sales_df["total_sales"] = sales_df["total_sales"].abs()

sales_df = sales_df.loc[sales_df["total_sales"] > 0]

sales_df = sales_df.sort_values(by=["date", "total_sales", "course_id"], ascending=[True, False, True], ignore_index=True)

## Data-quality validation

The prepared datasets are validated before export. The checks are executed separately so that each type of inconsistency can be identified clearly.

The validation sequence verifies:

1. uniqueness of records and primary identifiers;
2. absence of missing values;
3. absence of invalid negative numeric values;
4. absence of empty text fields;
5. chronological consistency between action start and end dates;
6. referential integrity between trainers, courses, actions, leads, sales, and the course catalogue.

A dataset only proceeds to the next analytical stage when all validations pass.

In [54]:
validate_duplicated(trainers_df, subset=["trainer_id"], df_name="trainers_df")
validate_duplicated(trainers_course_df, df_name="trainers_course_df")
validate_duplicated(trainers_local_df, df_name="trainers_local_df")
validate_duplicated(actions_df, df_name="actions_df")
validate_duplicated(info_df, subset=["course_id"], df_name="info_df")
validate_duplicated(leads_df, df_name="leads_df")
validate_duplicated(sales_df, df_name="sales_df")

print("All dataframes passed the duplicated rows validation")

All dataframes passed the duplicated rows validation


In [55]:
validate_missing_values(trainers_df, df_name="trainers_df")
validate_missing_values(trainers_course_df, df_name="trainers_course_df")
validate_missing_values(trainers_local_df, df_name="trainers_local_df")
validate_missing_values(actions_df, df_name="actions_df")
validate_missing_values(info_df, df_name="info_df")
validate_missing_values(leads_df, df_name="leads_df")
validate_missing_values(sales_df, df_name="sales_df")

print("All dataframes passed the missing values validation")

All dataframes passed the missing values validation


In [56]:
validate_negative_values(trainers_df, df_name="trainers_df")
validate_negative_values(trainers_course_df, df_name="trainers_course_df")
validate_negative_values(trainers_local_df, df_name="trainers_local_df")
validate_negative_values(actions_df, df_name="actions_df")
validate_negative_values(info_df, df_name="info_df")
validate_negative_values(leads_df, df_name="leads_df")
validate_negative_values(sales_df, df_name="sales_df")

print("All dataframes passed the negative values validation")

All dataframes passed the negative values validation


In [57]:
validate_empty_strings(trainers_df, df_name="trainers_df")
validate_empty_strings(trainers_course_df, df_name="trainers_course_df")
validate_empty_strings(trainers_local_df, df_name="trainers_local_df")
validate_empty_strings(actions_df, df_name="actions_df")
validate_empty_strings(info_df, df_name="info_df")
validate_empty_strings(leads_df, df_name="leads_df")
validate_empty_strings(sales_df, df_name="sales_df")

print("All dataframes passed the empty strings validation")

All dataframes passed the empty strings validation


In [58]:
validate_date_order(actions_df, start_col="start_date", end_col="end_date", df_name="actions_df")

print("All dataframes passed the date order validation")

All dataframes passed the date order validation


In [59]:
validate_foreign_key(trainers_course_df, column="trainer_id", ref_df=trainers_df, ref_column="trainer_id", df_name="trainers_course_df", ref_name="trainers_df")
validate_foreign_key(trainers_local_df, column="trainer_id", ref_df=trainers_df, ref_column="trainer_id", df_name="trainers_local_df", ref_name="trainers_df")

validate_foreign_key(trainers_course_df, column="course_id", ref_df=info_df, ref_column="course_id", df_name="trainers_course_df", ref_name="info_df")
validate_foreign_key(actions_df, column="course_id", ref_df=info_df, ref_column="course_id", df_name="actions_df", ref_name="info_df")
validate_foreign_key(leads_df, column="course_id", ref_df=info_df, ref_column="course_id", df_name="leads_df", ref_name="info_df")
validate_foreign_key(sales_df, column="course_id", ref_df=info_df, ref_column="course_id", df_name="sales_df", ref_name="info_df")

print("All foreign key validations passed.")

All foreign key validations passed.


## Processed dataset inspection

The structure and contents of each prepared dataset are inspected before export. The `.info()` outputs confirm row counts, column names, data types, and the absence of missing values, while the tabular previews allow a final visual review of the transformed records.

The trainer source table is retained for validation and inspection. The two normalised trainer-eligibility tables are exported for use in the analytical workflow.

In [60]:
trainers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   trainer_id  183 non-null    int64 
 1   course_id   183 non-null    object
 2   local       183 non-null    object
dtypes: int64(1), object(2)
memory usage: 4.4+ KB


In [61]:
trainers_df

,trainer_id,course_id,local
0,1,"[PBI, CYB]",[Centro 4]
1,2,"[PBI, CYB]",[Centro 2]
2,3,"[HAC, GER]",[Centro 5]
3,4,"[HAC, HCP, GER]",[Centro 2]
4,5,"[CYB, PYT]",[Centro 5]
...,...,...,...
178,179,[ING],[Centro 5]
179,180,[VND],[Centro 3]
180,181,[SOC],[Centro 2]
181,182,[HOT],[Centro 4]


In [62]:
trainers_course_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 335 entries, 0 to 334
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   trainer_id  335 non-null    int64
 1   course_id   335 non-null    str  
dtypes: int64(1), str(1)
memory usage: 5.4 KB


In [63]:
trainers_course_df

,trainer_id,course_id
0,1,CYB
1,1,PBI
2,2,CYB
3,2,PBI
4,3,GER
...,...,...
330,179,ING
331,180,VND
332,181,SOC
333,182,HOT


In [64]:
trainers_local_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   trainer_id  207 non-null    int64
 1   local       207 non-null    str  
dtypes: int64(1), str(1)
memory usage: 3.4 KB


In [65]:
trainers_local_df

,trainer_id,local
0,1,Centro 4
1,2,Centro 2
2,3,Centro 5
3,4,Centro 2
4,5,Centro 5
...,...,...
202,179,Centro 5
203,180,Centro 3
204,181,Centro 2
205,182,Centro 4


In [66]:
info_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   course_id    24 non-null     str  
 1   course_name  24 non-null     str  
 2   course_area  24 non-null     str  
 3   total_hours  24 non-null     int64
 4   active       24 non-null     bool 
dtypes: bool(1), int64(1), str(3)
memory usage: 924.0 bytes


In [67]:
info_df

,course_id,course_name,course_area,total_hours,active
0,CYB,Cibersegurança Básica,Informática,16,True
1,EMP,Empilhadores e Movimentação de Cargas,Segurança,16,True
2,EXC,Excel Aplicado à Gestão,Informática,20,True
3,FOR,Formação Pedagógica Inicial de Formadores,Formação,90,True
4,GER,Geriatria e Apoio ao Idoso,Saúde,50,True
5,GES,Gestão de Equipas,Gestão,16,True
6,HAC,Higiene e Segurança Alimentar,Alimentar,8,True
7,HCP,HACCP Aplicado,Alimentar,12,True
8,HOT,Housekeeping e Operações Hoteleiras,Turismo,25,True
9,INF,"Infeção, Prevenção e Controlo",Saúde,16,True


In [68]:
actions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19125 entries, 0 to 19124
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   action_id       19125 non-null  str           
 1   course_id       19125 non-null  str           
 2   local           19125 non-null  str           
 3   start_date      19125 non-null  datetime64[us]
 4   end_date        19125 non-null  datetime64[us]
 5   modality        19125 non-null  str           
 6   status          19125 non-null  str           
 7   total_students  19125 non-null  int64         
dtypes: datetime64[us](2), int64(1), str(5)
memory usage: 1.2 MB


In [69]:
actions_df

,action_id,course_id,local,start_date,end_date,modality,status,total_students
0,P_HCP_0001,HCP,Centro 1,2018-01-01,2018-01-02,In-person,Finalizada,11
1,P_INF_0001,INF,Centro 1,2018-01-01,2018-01-03,In-person,Finalizada,11
2,P_PRI_0001,PRI,Centro 1,2018-01-01,2018-01-03,In-person,Finalizada,15
3,H_TUR_0004,TUR,Centro 5,2018-01-01,2018-01-05,Hybrid,Finalizada,20
4,H_INF_0005,INF,Centro 5,2018-01-01,2018-01-06,Hybrid,Finalizada,20
...,...,...,...,...,...,...,...,...
19120,P_HAC_0778,HAC,Centro 4,2026-12-30,2026-12-31,In-person,Prevista,25
19121,H_ING_0583,ING,Centro 2,2026-12-30,2026-12-31,Hybrid,Prevista,21
19122,E_PRO_0618,PRO,Centro 4,2026-12-30,2026-12-31,Online,Prevista,31
19123,P_SCI_0684,SCI,Centro 3,2026-12-30,2026-12-31,In-person,Prevista,21


In [70]:
leads_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23973 entries, 0 to 23972
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   course_id    23973 non-null  str           
 1   local        23973 non-null  str           
 2   modality     23973 non-null  str           
 3   date         23973 non-null  datetime64[us]
 4   total_leads  23973 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 936.6 KB


In [71]:
leads_df

,course_id,local,modality,date,total_leads
0,FOR,Centro 2,Hybrid,2023-01-02,54
1,EXC,Centro 4,Online,2023-01-02,41
2,PYT,Centro 1,Online,2023-01-02,39
3,GES,Centro 4,Online,2023-01-02,36
4,GES,Centro 4,Hybrid,2023-01-02,36
...,...,...,...,...,...
23968,GER,Centro 2,Hybrid,2026-12-21,18
23969,PRO,Centro 4,Online,2026-12-21,8
23970,HOT,Centro 5,In-person,2026-12-21,6
23971,PBI,Centro 4,Online,2026-12-21,4


In [72]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23941 entries, 0 to 23940
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   course_id    23941 non-null  str           
 1   local        23941 non-null  str           
 2   modality     23941 non-null  str           
 3   date         23941 non-null  datetime64[us]
 4   total_sales  23941 non-null  int64         
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 935.3 KB


In [73]:
sales_df

,course_id,local,modality,date,total_sales
0,FOR,Centro 2,Hybrid,2023-01-02,20
1,EXC,Centro 4,Online,2023-01-02,18
2,LOG,Centro 4,In-person,2023-01-02,15
3,GES,Centro 4,Online,2023-01-02,14
4,GES,Centro 4,Hybrid,2023-01-02,14
...,...,...,...,...,...
23936,SOC,Centro 5,Hybrid,2026-12-21,5
23937,HOT,Centro 5,In-person,2026-12-21,2
23938,PRO,Centro 4,Online,2026-12-21,2
23939,PBI,Centro 4,Online,2026-12-21,1


## Export processed data

The validated datasets are exported to `data/processed/` using UTF-8 encoding.

The resulting files provide the inputs for the subsequent notebooks:

- `trainers_course.csv`: trainer eligibility by course;
- `trainers_local.csv`: trainer eligibility by location;
- `training_actions.csv`: standardised training activity;
- `info_actions.csv`: course reference information;
- `leads.csv`: observed commercial leads;
- `sales.csv`: observed commercial sales.

Separating processed data from the raw source files preserves the original datasets and makes the analytical workflow easier to reproduce and audit.

In [74]:
trainers_course_df.to_csv(PROCESSED_DATA_DIR / "trainers_course.csv", sep=",", encoding="utf-8", index=False)

trainers_local_df.to_csv(PROCESSED_DATA_DIR / "trainers_local.csv", sep=",", encoding="utf-8", index=False)

actions_df.to_csv(PROCESSED_DATA_DIR / "training_actions.csv", sep=",", encoding="utf-8", index=False)

info_df.to_csv(PROCESSED_DATA_DIR / "info_actions.csv", sep=",", encoding="utf-8", index=False)

leads_df.to_csv(PROCESSED_DATA_DIR / "leads.csv", sep=",", encoding="utf-8", index=False)

sales_df.to_csv(PROCESSED_DATA_DIR / "sales.csv", sep=",", encoding="utf-8", index=False)

## Preparation outcome

The data-preparation stage produced a validated and standardised analytical base containing:

- 183 trainers;
- 335 trainer–course relationships;
- 207 trainer–location relationships;
- 24 courses;
- 19,125 training actions;
- 23,973 lead records;
- 23,941 sales records.

All datasets passed the duplicate, missing-value, negative-value, empty-string, date-order, and foreign-key validations.

The processed files can now be used to analyse workforce vulnerability, operational activity, commercial pressure, hiring priority, and trainer-capacity requirements.